# 02 EDA

- 목표: 데이터 구조 파악 및 피처·타겟 관계 시각화
- 데이터: BRFSS Diabetes Health Indicators (253,680행, 21 피처)
- 타겟: `Diabetes_binary` (0: 정상, 1: 당뇨/전당뇨)
- 우선순위: Recall 최대화 (미탐지 비용 > 오탐지 비용)
- 출력: `outputs/figures/`에 시각화 결과 저장

## 섹션 1 — 데이터 로드 및 기초 정보 확인

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

DATA_PATH = Path('../data/diabetes_indicators.csv')
FIGURE_DIR = Path('../outputs/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
feature_cols = [col for col in df.columns if col != 'Diabetes_binary']

print('데이터 형태:', df.shape)
print('\n데이터 타입:')
print(df.dtypes)
print('\n기초 통계:')
display(df.describe())
print('\n상위 5행:')
display(df.head())

## 섹션 2 — 타겟 클래스 분포

In [ ]:
counts = df['Diabetes_binary'].value_counts().sort_index()
ratios = df['Diabetes_binary'].value_counts(normalize=True).sort_index()
class_summary = pd.DataFrame({
    '건수': counts.astype(int),
    '비율': ratios,
    '비율(%)': ratios * 100,
})
class_summary.index = class_summary.index.map({0.0: '0 - 정상', 1.0: '1 - 당뇨/전당뇨'})
display(class_summary)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(class_summary.index, class_summary['건수'], color=['steelblue', 'darkorange'])
ax.set_title('타겟 클래스 분포')
ax.set_xlabel('Diabetes_binary')
ax.set_ylabel('건수')
ax.bar_label(
    bars,
    labels=[f'{count:,}\n({pct:.1f}%)' for count, pct in zip(class_summary['건수'], class_summary['비율(%)'])],
    padding=3,
)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 섹션 3 — 피처별 분포 히스토그램

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(20, 16))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    df[col].hist(ax=axes_flat[i], bins=30, color='steelblue', edgecolor='white')
    axes_flat[i].set_title(col)
    axes_flat[i].set_xlabel(col)
    axes_flat[i].set_ylabel('빈도')

for j in range(len(feature_cols), len(axes_flat)):
    axes_flat[j].axis('off')

fig.suptitle('피처 분포', y=1.01, fontsize=16)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 섹션 4 — 타겟별 피처 분포 비교

In [ ]:
continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'Age', 'Income', 'Education']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(continuous_features):
    ax = axes.flatten()[i]
    df.boxplot(column=col, by='Diabetes_binary', ax=ax, grid=False)
    ax.set_title(col)
    ax.set_xlabel('Diabetes_binary')
    ax.set_ylabel(col)

fig.suptitle('당뇨 여부별 피처 분포')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_feature_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 섹션 5 — 상관관계 히트맵

In [ ]:
corr = df.corr(numeric_only=True)

top_corr = (
    corr['Diabetes_binary']
    .drop('Diabetes_binary')
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .head(5)
)
print('Diabetes_binary 상관관계 상위 5개 피처:')
display(top_corr.to_frame(name='상관계수'))

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.3, ax=ax)
ax.set_title('피처 상관관계 히트맵')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 섹션 6 — 의학적 해석 메모

| 피처 | 당뇨와의 관계 |
|------|---------------|
| BMI | BMI가 높을수록, 특히 비만 수준에서 인슐린 저항성과 당뇨 위험이 크게 증가한다. |
| HighBP | 고혈압과 당뇨는 공통 심혈관 대사 위험 경로를 통해 자주 동반 발생한다. |
| HighChol | 이상지질혈증은 대사 증후군 및 당뇨 위험과 밀접하게 연관된다. |
| Smoker | 흡연은 인슐린 감수성을 저하시키고 심혈관 위험을 악화시킬 수 있다. |
| PhysActivity | 신체 활동 부족은 당뇨의 수정 가능한 위험 요인이다. |
| Age | 당뇨 위험은 나이가 들수록 증가하며, 특히 중년 이후에 급격히 높아진다. |
| GenHlth | 전반적 건강 상태 자기 평가가 낮을수록 만성 질환 부담을 반영한다. |
| HvyAlcoholConsump | 과도한 음주는 간 기능, 체중, 혈당 조절에 영향을 미칠 수 있다. |

## 섹션 7 — 이상치 후보 및 결측치 확인

In [ ]:
bmi_min = df['BMI'].min()
bmi_max = df['BMI'].max()
bmi_outlier_count = (df['BMI'] > 60).sum()
bmi_zero_count = (df['BMI'] == 0).sum()
missing_by_col = df.isnull().sum()
total_missing = missing_by_col.sum()

print(f'BMI 범위: {bmi_min} ~ {bmi_max}')
print(f'BMI 60 초과 행 수: {bmi_outlier_count}')
print(f'BMI 0값 개수: {bmi_zero_count}')
print('\n컬럼별 결측치 수:')
print(missing_by_col)
print(f'\n전체 결측치 수: {total_missing}')

### 이상치 및 결측치 확인 결과

- BMI 범위: 12.0 ~ 98.0
- BMI 60 초과: 805행
- BMI 0값: 없음
- 전체 결측치: 0

전략 선택 (Claude 검토 후 기재):
- 이상치: Z-score (threshold=3) 클리핑 적용
- 결측치: Median 대체 (파이프라인 안전망)
- 클래스 불균형: SMOTE + 언더샘플링 결합